# 🌍 Tourism Experience Analytics
## Classification, Prediction, and Recommendation System

**Domain:** Tourism | **Project Type:** Machine Learning (Regression + Classification + Recommendation)

---

### 📋 Project Overview
This project analyzes user preferences, travel patterns, and attraction features to achieve three primary objectives:
1. **Regression** — Predict the rating a user will give to a tourist attraction
2. **Classification** — Predict the visit mode (Business, Family, Couples, Friends, etc.)
3. **Recommendation** — Suggest personalized tourist attractions based on user history

### 📂 Dataset Tables
| Table | Description |
|---|---|
| Transaction | User visits with ratings, visit mode, year/month |
| User | User demographics (continent, region, country, city) |
| Item | Tourist attractions with type and location |
| Mode | Visit mode lookup table |
| Type | Attraction type lookup table |
| City / Country / Region / Continent | Geographic hierarchies |


## Step 1 — Import Libraries & Setup

In [1]:
# Standard libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ML libraries
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (mean_squared_error, r2_score, mean_absolute_error,
                             accuracy_score, classification_report, confusion_matrix,
                             precision_score, recall_score, f1_score)
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import TruncatedSVD
from scipy.sparse import csr_matrix

import xgboost as xgb
import lightgbm as lgb

# Plotting style
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12
sns.set_style('whitegrid')
sns.set_palette('Set2')

print("✅ All libraries loaded successfully!")


✅ All libraries loaded successfully!


## Step 2 — Load All Datasets

In [2]:
# ── Load all tables ──────────────────────────────────────────────────
transaction = pd.read_excel('Transaction.xlsx')
user        = pd.read_excel('User.xlsx')
item        = pd.read_excel('Item.xlsx')
mode        = pd.read_excel('Mode.xlsx')
atype       = pd.read_excel('Type.xlsx')
city        = pd.read_excel('City.xlsx')
country     = pd.read_excel('Country.xlsx')
region      = pd.read_excel('Region.xlsx')
continent   = pd.read_excel('Continent.xlsx')

tables = {
    'Transaction': transaction, 'User': user, 'Item': item,
    'Mode': mode, 'AttractionType': atype, 'City': city,
    'Country': country, 'Region': region, 'Continent': continent
}

print("=" * 55)
print(f"{'Table':<18} {'Rows':>8} {'Columns':>10}")
print("=" * 55)
for name, df in tables.items():
    print(f"{name:<18} {df.shape[0]:>8,} {df.shape[1]:>10}")
print("=" * 55)


Table                  Rows    Columns
Transaction          52,930          7
User                 33,530          5
Item                     30          5
Mode                      6          2
AttractionType           17          2
City                  9,143          3
Country                 165          3
Region                   22          3
Continent                 6          2


In [3]:
# Preview each table
for name, df in tables.items():
    print(f"\n{'─'*50}")
    print(f"📄  {name}")
    print(f"{'─'*50}")
    display(df.head(3))



──────────────────────────────────────────────────
📄  Transaction
──────────────────────────────────────────────────


,TransactionId,UserId,VisitYear,VisitMonth,VisitMode,AttractionId,Rating
0,3,70456,2022,10,2,640,5
1,8,7567,2022,10,4,640,5
2,9,79069,2022,10,3,640,5



──────────────────────────────────────────────────
📄  User
──────────────────────────────────────────────────


,UserId,ContinentId,RegionId,CountryId,CityId
0,14,5,20,155,220.0
1,16,3,14,101,3098.0
2,20,4,15,109,4303.0



──────────────────────────────────────────────────
📄  Item
──────────────────────────────────────────────────


...

,CountryId,Country,RegionId
0,0,-,0
1,1,Cameroon,1
2,2,Chad,1



──────────────────────────────────────────────────
📄  Region
──────────────────────────────────────────────────


,Region,RegionId,ContinentId
0,-,0,0
1,Central Africa,1,1
2,East Africa,2,1



──────────────────────────────────────────────────
📄  Continent
──────────────────────────────────────────────────


,ContinentId,Continent
0,0,-
1,1,Africa
2,2,America


## Step 3 — Data Understanding & Initial Inspection

In [4]:
# ── Transaction table: main driver ──────────────────────────────────
print("=== Transaction ===")
print(transaction.info())
print("\nDescriptive Stats:")
display(transaction.describe())


=== Transaction ===
<class 'pandas.DataFrame'>
RangeIndex: 52930 entries, 0 to 52929
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   TransactionId  52930 non-null  int64
 1   UserId         52930 non-null  int64
 2   VisitYear      52930 non-null  int64
 3   VisitMonth     52930 non-null  int64
 4   VisitMode      52930 non-null  int64
 5   AttractionId   52930 non-null  int64
 6   Rating         52930 non-null  int64
dtypes: int64(7)
memory usage: 2.8 MB
None

Descriptive Stats:


,TransactionId,UserId,VisitYear,VisitMonth,VisitMode,AttractionId,Rating
count,52930.000000,52930.000000,52930.000000,52930.000000,52930.000000,52930.000000,52930.000000
mean,70415.130474,45024.522596,2016.351899,6.633100,2.945759,759.663782,4.157699
std,66299.514206,25073.062199,1.732926,3.392979,1.000683,210.716111,0.970543
min,3.000000,14.000000,2013.000000,1.000000,1.000000,369.000000,1.000000
25%,19646.250000,23470.000000,2015.000000,4.000000,2.000000,640.000000,4.000000
50%,42770.500000,45533.500000,2016.000000,7.000000,3.000000,737.000000,4.000000
75%,105638.750000,66667.000000,2018.000000,9.000000,4.000000,841.000000,5.000000
max,211241.000000,88190.000000,2022.000000,12.000000,5.000000,1297.000000,5.000000


In [5]:
# ── Check value ranges ───────────────────────────────────────────────
print("Rating value counts:")
print(transaction['Rating'].value_counts().sort_index())
print("\nVisitMode value counts:")
print(transaction['VisitMode'].value_counts())
print("\nVisitYear range:", transaction['VisitYear'].min(), "–", transaction['VisitYear'].max())
print("VisitMonth range:", transaction['VisitMonth'].min(), "–", transaction['VisitMonth'].max())


Rating value counts:
Rating
1     1263
2     2035
3     7730
4    17966
5    23936
Name: count, dtype: int64

VisitMode value counts:
VisitMode
2    21620
3    15217
4    10945
5     4525
1      623
Name: count, dtype: int64

VisitYear range: 2013 – 2022
VisitMonth range: 1 – 12


## Step 4 — Data Cleaning

### 4.1 Handle Missing Values

In [6]:
# ── Check nulls in every table ───────────────────────────────────────
print("Missing values per table:")
print("=" * 40)
for name, df in tables.items():
    nulls = df.isnull().sum().sum()
    print(f"  {name:<18}: {nulls} null(s)")


Missing values per table:
  Transaction       : 0 null(s)
  User              : 4 null(s)
  Item              : 0 null(s)
  Mode              : 0 null(s)
  AttractionType    : 0 null(s)
  City              : 1 null(s)
  Country           : 0 null(s)
  Region            : 0 null(s)
  Continent         : 0 null(s)


In [7]:
# ── Fix User table – CityId has nulls ────────────────────────────────
print("User – CityId nulls:", user['CityId'].isnull().sum())

# Fill missing CityId with 0 (unknown)
user['CityId'] = user['CityId'].fillna(0).astype(int)
print("After fill:", user['CityId'].isnull().sum(), "nulls remaining")


User – CityId nulls: 4
After fill: 0 nulls remaining


### 4.2 Remove Placeholder / Unknown Rows

In [8]:
# Rows with Id == 0 are placeholder 'unknown' entries in lookup tables
city      = city[city['CityId'] != 0].reset_index(drop=True)
country   = country[country['CountryId'] != 0].reset_index(drop=True)
region    = region[region['RegionId'] != 0].reset_index(drop=True)
continent = continent[continent['ContinentId'] != 0].reset_index(drop=True)
mode      = mode[mode['VisitModeId'] != 0].reset_index(drop=True)

print("Cleaned shapes:")
for name, df in [('City', city), ('Country', country), ('Region', region),
                  ('Continent', continent), ('Mode', mode)]:
    print(f"  {name}: {df.shape}")


Cleaned shapes:
  City: (9142, 3)
  Country: (164, 3)
  Region: (21, 3)
  Continent: (5, 2)
  Mode: (5, 2)


### 4.3 Check & Fix Duplicates

In [9]:
# Duplicate check across all tables
for name, df in tables.items():
    dups = df.duplicated().sum()
    if dups > 0:
        print(f"  ⚠️  {name}: {dups} duplicates")
    else:
        print(f"  ✅ {name}: no duplicates")

# Drop duplicates if any
transaction.drop_duplicates(inplace=True)
user.drop_duplicates(inplace=True)
print("\nDuplicate removal complete.")


  ✅ Transaction: no duplicates
  ✅ User: no duplicates
  ✅ Item: no duplicates
  ✅ Mode: no duplicates
  ✅ AttractionType: no duplicates
  ✅ City: no duplicates
  ✅ Country: no duplicates
  ✅ Region: no duplicates
  ✅ Continent: no duplicates

Duplicate removal complete.


### 4.4 Outlier Detection in Rating

In [10]:
# Rating should be between 1 and 5
invalid_ratings = transaction[~transaction['Rating'].between(1, 5)]
print(f"Ratings outside [1,5]: {len(invalid_ratings)}")

# Filter to valid ratings only
transaction = transaction[transaction['Rating'].between(1, 5)].reset_index(drop=True)
print(f"Transaction rows after filter: {len(transaction):,}")

# Box plot of ratings
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
transaction['Rating'].hist(bins=5, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Rating Distribution')
axes[0].set_xlabel('Rating')
axes[0].set_ylabel('Count')

transaction.boxplot(column='Rating', ax=axes[1])
axes[1].set_title('Rating Boxplot')
plt.tight_layout()
plt.show()


Ratings outside [1,5]: 0
Transaction rows after filter: 52,930


## Step 5 — Data Merging & Feature Engineering

### 5.1 Build Consolidated Dataset

In [11]:
# ── Start with Transaction ───────────────────────────────────────────
df = transaction.copy()

# Merge VisitMode name
df = df.merge(mode.rename(columns={'VisitModeId': 'VisitMode', 'VisitMode': 'VisitModeName'}),
              on='VisitMode', how='left')

# Merge User demographics
df = df.merge(user, on='UserId', how='left')

# Merge Continent
df = df.merge(continent.rename(columns={'ContinentId': 'ContinentId', 'Continent': 'ContinentName'}),
              on='ContinentId', how='left')

# Merge Region
df = df.merge(region[['RegionId', 'Region']].rename(columns={'Region': 'RegionName'}),
              on='RegionId', how='left')

# Merge Country
df = df.merge(country[['CountryId', 'Country']].rename(columns={'Country': 'CountryName'}),
              on='CountryId', how='left')

# Merge City
df = df.merge(city[['CityId', 'CityName']].rename(columns={'CityName': 'UserCityName'}),
              on='CityId', how='left')

# Merge Item (Attraction)
df = df.merge(item, on='AttractionId', how='left')

# Merge Attraction Type
df = df.merge(atype, on='AttractionTypeId', how='left')

# Merge Attraction City
df = df.merge(city[['CityId', 'CityName']].rename(columns={'CityId': 'AttractionCityId',
                                                             'CityName': 'AttractionCityName'}),
              on='AttractionCityId', how='left')

print(f"Consolidated dataset shape: {df.shape}")
print("\nColumns:", list(df.columns))


Consolidated dataset shape: (52930, 22)

Columns: ['TransactionId', 'UserId', 'VisitYear', 'VisitMonth', 'VisitMode', 'AttractionId', 'Rating', 'VisitModeName', 'ContinentId', 'RegionId', 'CountryId', 'CityId', 'ContinentName', 'RegionName', 'CountryName', 'UserCityName', 'AttractionCityId', 'AttractionTypeId', 'Attraction', 'AttractionAddress', 'AttractionType', 'AttractionCityName']


In [12]:
display(df.head(3))
print("\nNulls in consolidated df:")
print(df.isnull().sum()[df.isnull().sum() > 0])


,TransactionId,UserId,VisitYear,VisitMonth,VisitMode,AttractionId,Rating,VisitModeName,ContinentId,RegionId,...,ContinentName,RegionName,CountryName,UserCityName,AttractionCityId,AttractionTypeId,Attraction,AttractionAddress,AttractionType,AttractionCityName
0,3,70456,2022,10,2,640,5,Couples,5,21,...,Europe,Western Europe,United Kingdom,Guildford,1,63,Sacred Monkey Forest Sanctuary,"Jl. Monkey Forest, Ubud 80571 Indonesia",Nature & Wildlife Areas,Douala
1,8,7567,2022,10,4,640,5,Friends,2,8,...,America,Northern America,Canada,Ontario,1,63,Sacred Monkey Forest Sanctuary,"Jl. Monkey Forest, Ubud 80571 Indonesia",Nature & Wildlife Areas,Douala
2,9,79069,2022,10,3,640,5,Family,2,9,...,America,South America,Brazil,Brazil,1,63,Sacred Monkey Forest Sanctuary,"Jl. Monkey Forest, Ubud 80571 Indonesia",Nature & Wildlife Areas,Douala



Nulls in consolidated df:
RegionName      32
UserCityName     8
dtype: int64


### 5.2 Feature Engineering

In [13]:
# ── Season from Visit Month ───────────────────────────────────────────
def month_to_season(m):
    if m in [12, 1, 2]:  return 'Winter'
    elif m in [3, 4, 5]: return 'Spring'
    elif m in [6, 7, 8]: return 'Summer'
    else:                return 'Autumn'

df['Season'] = df['VisitMonth'].apply(month_to_season)

# ── Avg Rating per Attraction ─────────────────────────────────────────
attraction_avg = df.groupby('AttractionId')['Rating'].mean().rename('AttractionAvgRating')
df = df.join(attraction_avg, on='AttractionId')

# ── Avg Rating per User ───────────────────────────────────────────────
user_avg = df.groupby('UserId')['Rating'].mean().rename('UserAvgRating')
df = df.join(user_avg, on='UserId')

# ── Visit Count per User ──────────────────────────────────────────────
visit_count = df.groupby('UserId')['TransactionId'].count().rename('UserVisitCount')
df = df.join(visit_count, on='UserId')

# ── Visit Count per Attraction ────────────────────────────────────────
att_pop = df.groupby('AttractionId')['TransactionId'].count().rename('AttractionPopularity')
df = df.join(att_pop, on='AttractionId')

print("New features added:")
print("  ✅ Season")
print("  ✅ AttractionAvgRating")
print("  ✅ UserAvgRating")
print("  ✅ UserVisitCount")
print("  ✅ AttractionPopularity")
print(f"\nFinal shape: {df.shape}")


New features added:
  ✅ Season
  ✅ AttractionAvgRating
  ✅ UserAvgRating
  ✅ UserVisitCount
  ✅ AttractionPopularity

Final shape: (52930, 27)


### 5.3 Encoding Categorical Variables

In [14]:
# ── Label encoding for ML models ─────────────────────────────────────
label_cols = ['VisitModeName', 'ContinentName', 'RegionName', 'CountryName',
              'AttractionType', 'Season']

le_dict = {}
for col in label_cols:
    df[col] = df[col].fillna('Unknown')
    le = LabelEncoder()
    df[col + '_enc'] = le.fit_transform(df[col])
    le_dict[col] = le

print("Encoded columns:")
for col in label_cols:
    n = df[col].nunique()
    print(f"  {col}: {n} unique categories")


Encoded columns:
  VisitModeName: 5 unique categories
  ContinentName: 5 unique categories
  RegionName: 22 unique categories
  CountryName: 153 unique categories
  AttractionType: 17 unique categories
  Season: 4 unique categories


## Step 6 — Exploratory Data Analysis (EDA)

### 6.1 User Distribution by Continent

In [15]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Continent distribution
cont_counts = df.drop_duplicates('UserId')['ContinentName'].value_counts()
cont_counts.plot(kind='bar', ax=axes[0], color=sns.color_palette('Set2', len(cont_counts)))
axes[0].set_title('Users by Continent')
axes[0].set_xlabel('Continent')
axes[0].set_ylabel('Number of Users')
axes[0].tick_params(axis='x', rotation=30)

# Top 10 regions
reg_counts = df.drop_duplicates('UserId')['RegionName'].value_counts().head(10)
reg_counts.plot(kind='barh', ax=axes[1], color='steelblue')
axes[1].set_title('Top 10 Regions by User Count')
axes[1].set_xlabel('Number of Users')

plt.tight_layout()
plt.show()

print("Continent breakdown:")
print(cont_counts.to_string())


Continent breakdown:
ContinentName
Australia & Oceania    9837
Asia                   9822
Europe                 8260
America                5018
Africa                  593


### 6.2 Visit Mode Analysis

In [16]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count
mode_counts = df['VisitModeName'].value_counts()
axes[0].bar(mode_counts.index, mode_counts.values,
            color=sns.color_palette('pastel', len(mode_counts)))
axes[0].set_title('Transaction Count by Visit Mode')
axes[0].set_xlabel('Visit Mode')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=20)

# Avg rating per visit mode
mode_rating = df.groupby('VisitModeName')['Rating'].mean().sort_values(ascending=False)
axes[1].bar(mode_rating.index, mode_rating.values,
            color=sns.color_palette('Set3', len(mode_rating)))
axes[1].set_title('Average Rating by Visit Mode')
axes[1].set_xlabel('Visit Mode')
axes[1].set_ylabel('Avg Rating')
axes[1].set_ylim(0, 5.5)
axes[1].axhline(df['Rating'].mean(), color='red', linestyle='--', label='Overall Mean')
axes[1].legend()
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()

print("\nAverage rating per visit mode:")
print(mode_rating.round(3).to_string())



Average rating per visit mode:
VisitModeName
Business    4.313
Family      4.219
Friends     4.174
Couples     4.117
Solo        4.088


### 6.3 Attraction Type Popularity

In [17]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Transaction count per type
type_counts = df['AttractionType'].value_counts().head(12)
type_counts.plot(kind='barh', ax=axes[0], color='teal')
axes[0].set_title('Top 12 Attraction Types by Visits')
axes[0].set_xlabel('Number of Visits')

# Avg rating per type
type_rating = df.groupby('AttractionType')['Rating'].mean().sort_values(ascending=False).head(12)
type_rating.plot(kind='barh', ax=axes[1], color='coral')
axes[1].set_title('Avg Rating per Attraction Type (Top 12)')
axes[1].set_xlabel('Average Rating')
axes[1].axvline(df['Rating'].mean(), color='navy', linestyle='--', label='Overall Mean')
axes[1].legend()

plt.tight_layout()
plt.show()


### 6.4 Temporal Trends

In [18]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Monthly trends
monthly = df.groupby('VisitMonth').agg(Visits=('TransactionId','count'),
                                        AvgRating=('Rating','mean')).reset_index()
month_labels = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
axes[0].bar(monthly['VisitMonth'], monthly['Visits'], color='steelblue')
axes[0].set_xticks(range(1,13))
axes[0].set_xticklabels(month_labels, rotation=30)
axes[0].set_title('Visits by Month')
axes[0].set_ylabel('Number of Visits')

axes[1].plot(monthly['VisitMonth'], monthly['AvgRating'], marker='o', color='orangered')
axes[1].set_xticks(range(1,13))
axes[1].set_xticklabels(month_labels, rotation=30)
axes[1].set_title('Average Rating by Month')
axes[1].set_ylabel('Avg Rating')
axes[1].set_ylim(0, 5.5)

plt.tight_layout()
plt.show()

# Yearly volume
yearly = df.groupby('VisitYear').size()
print("Yearly transaction volume:")
print(yearly.to_string())


Yearly transaction volume:
VisitYear
2013     2983
2014     4808
2015     8687
2016    12823
2017     9444
2018     7461
2019     5913
2020      529
2021       35
2022      247


### 6.5 Rating Distribution by Continent

In [19]:
plt.figure(figsize=(14, 5))
cont_order = df.groupby('ContinentName')['Rating'].mean().sort_values(ascending=False).index
sns.boxplot(data=df, x='ContinentName', y='Rating', order=cont_order, palette='Set2')
plt.title('Rating Distribution by Continent')
plt.xlabel('Continent')
plt.ylabel('Rating')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


### 6.6 Correlation Analysis

In [20]:
num_cols = ['Rating', 'VisitYear', 'VisitMonth', 'AttractionAvgRating',
            'UserAvgRating', 'UserVisitCount', 'AttractionPopularity']

corr = df[num_cols].corr()

plt.figure(figsize=(10, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', mask=mask,
            linewidths=0.5, vmin=-1, vmax=1)
plt.title('Correlation Heatmap — Numerical Features')
plt.tight_layout()
plt.show()

print("\nTop correlations with Rating:")
print(corr['Rating'].drop('Rating').abs().sort_values(ascending=False).round(4))



Top correlations with Rating:
UserAvgRating           0.8538
AttractionAvgRating     0.3019
AttractionPopularity    0.1217
UserVisitCount          0.0295
VisitMonth              0.0192
VisitYear               0.0063
Name: Rating, dtype: float64


### 6.7 Season-wise Analysis

In [21]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

season_counts = df['Season'].value_counts()
axes[0].pie(season_counts, labels=season_counts.index, autopct='%1.1f%%',
            colors=sns.color_palette('pastel'))
axes[0].set_title('Visit Distribution by Season')

season_rating = df.groupby('Season')['Rating'].mean().sort_values(ascending=False)
axes[1].bar(season_rating.index, season_rating.values,
            color=sns.color_palette('Set2', 4))
axes[1].set_title('Average Rating by Season')
axes[1].set_ylabel('Avg Rating')
axes[1].set_ylim(0, 5.5)

plt.tight_layout()
plt.show()


### 6.8 Top Attractions by Visits & Rating

In [22]:
top_att = (df.groupby(['AttractionId', 'Attraction'])
             .agg(Visits=('TransactionId','count'), AvgRating=('Rating','mean'))
             .sort_values('Visits', ascending=False)
             .head(15)
             .reset_index())

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

axes[0].barh(top_att['Attraction'], top_att['Visits'], color='cadetblue')
axes[0].set_title('Top 15 Attractions by Number of Visits')
axes[0].set_xlabel('Visits')
axes[0].invert_yaxis()

top_rated = (df.groupby(['AttractionId', 'Attraction'])
               .agg(Visits=('TransactionId','count'), AvgRating=('Rating','mean'))
               .query('Visits >= 50')
               .sort_values('AvgRating', ascending=False)
               .head(15)
               .reset_index())
axes[1].barh(top_rated['Attraction'], top_rated['AvgRating'], color='salmon')
axes[1].set_title('Top 15 Attractions by Avg Rating (min 50 visits)')
axes[1].set_xlabel('Avg Rating')
axes[1].set_xlim(0, 5.5)
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()


## Step 7 — ML Feature Preparation

In [23]:
# ── Feature set for both Regression & Classification ─────────────────
feature_cols = [
    'VisitYear', 'VisitMonth',
    'ContinentName_enc', 'RegionName_enc', 'AttractionType_enc',
    'Season_enc', 'AttractionAvgRating', 'UserAvgRating',
    'UserVisitCount', 'AttractionPopularity', 'AttractionTypeId'
]

# Drop rows with any NaN in features or targets
ml_df = df[feature_cols + ['Rating', 'VisitModeName_enc', 'VisitModeName']].dropna()

X = ml_df[feature_cols]
y_reg   = ml_df['Rating']           # Regression target
y_clf   = ml_df['VisitModeName_enc']  # Classification target

print(f"ML-ready samples: {len(ml_df):,}")
print(f"Feature count   : {len(feature_cols)}")
print(f"Regression target (Rating) — unique values: {y_reg.nunique()}")
print(f"Classification target (VisitMode) — classes: {y_clf.nunique()}")
print("\nClass distribution (VisitMode):")
print(ml_df['VisitModeName'].value_counts())


ML-ready samples: 52,930
Feature count   : 11
Regression target (Rating) — unique values: 5
Classification target (VisitMode) — classes: 5

Class distribution (VisitMode):
VisitModeName
Couples     21620
Family      15217
Friends     10945
Solo         4525
Business      623
Name: count, dtype: int64


In [24]:
# ── Train / Test Split ────────────────────────────────────────────────
X_train, X_test, yr_train, yr_test, yc_train, yc_test = train_test_split(
    X, y_reg, y_clf, test_size=0.2, random_state=42, stratify=y_clf)

print(f"Train size: {X_train.shape[0]:,}")
print(f"Test  size: {X_test.shape[0]:,}")

# Scale features
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)


Train size: 42,344
Test  size: 10,586


## Step 8 — Regression: Predicting Attraction Ratings

### 8.1 Baseline — Linear Regression

In [25]:
lr = LinearRegression()
lr.fit(X_train_s, yr_train)
yr_pred_lr = lr.predict(X_test_s)

rmse_lr = np.sqrt(mean_squared_error(yr_test, yr_pred_lr))
mae_lr  = mean_absolute_error(yr_test, yr_pred_lr)
r2_lr   = r2_score(yr_test, yr_pred_lr)

print("── Linear Regression ──────────────────────")
print(f"  RMSE : {rmse_lr:.4f}")
print(f"  MAE  : {mae_lr:.4f}")
print(f"  R²   : {r2_lr:.4f}")


── Linear Regression ──────────────────────
  RMSE : 0.5034
  MAE  : 0.2950
  R²   : 0.7341


### 8.2 Random Forest Regressor

In [26]:
rfr = RandomForestRegressor(n_estimators=150, random_state=42, n_jobs=-1)
rfr.fit(X_train, yr_train)
yr_pred_rf = rfr.predict(X_test)

rmse_rf = np.sqrt(mean_squared_error(yr_test, yr_pred_rf))
mae_rf  = mean_absolute_error(yr_test, yr_pred_rf)
r2_rf   = r2_score(yr_test, yr_pred_rf)

print("── Random Forest Regressor ─────────────────")
print(f"  RMSE : {rmse_rf:.4f}")
print(f"  MAE  : {mae_rf:.4f}")
print(f"  R²   : {r2_rf:.4f}")


── Random Forest Regressor ─────────────────
  RMSE : 0.5449
  MAE  : 0.2788
  R²   : 0.6885


### 8.3 XGBoost Regressor

In [27]:
xgbr = xgb.XGBRegressor(n_estimators=200, learning_rate=0.05,
                        max_depth=6, random_state=42, verbosity=0)
xgbr.fit(X_train, yr_train)
yr_pred_xgb = xgbr.predict(X_test)

rmse_xgb = np.sqrt(mean_squared_error(yr_test, yr_pred_xgb))
mae_xgb  = mean_absolute_error(yr_test, yr_pred_xgb)
r2_xgb   = r2_score(yr_test, yr_pred_xgb)

print("── XGBoost Regressor ───────────────────────")
print(f"  RMSE : {rmse_xgb:.4f}")
print(f"  MAE  : {mae_xgb:.4f}")
print(f"  R²   : {r2_xgb:.4f}")


── XGBoost Regressor ───────────────────────
  RMSE : 0.4977
  MAE  : 0.2674
  R²   : 0.7401


### 8.4 LightGBM Regressor

In [28]:
lgbr = lgb.LGBMRegressor(n_estimators=200, learning_rate=0.05,
                        num_leaves=63, random_state=42, verbose=-1)
lgbr.fit(X_train, yr_train)
yr_pred_lgb = lgbr.predict(X_test)

rmse_lgb = np.sqrt(mean_squared_error(yr_test, yr_pred_lgb))
mae_lgb  = mean_absolute_error(yr_test, yr_pred_lgb)
r2_lgb   = r2_score(yr_test, yr_pred_lgb)

print("── LightGBM Regressor ──────────────────────")
print(f"  RMSE : {rmse_lgb:.4f}")
print(f"  MAE  : {mae_lgb:.4f}")
print(f"  R²   : {r2_lgb:.4f}")


── LightGBM Regressor ──────────────────────
  RMSE : 0.4974
  MAE  : 0.2647
  R²   : 0.7405


### 8.5 Regression Model Comparison

In [29]:
reg_results = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest', 'XGBoost', 'LightGBM'],
    'RMSE':  [rmse_lr, rmse_rf, rmse_xgb, rmse_lgb],
    'MAE':   [mae_lr,  mae_rf,  mae_xgb,  mae_lgb],
    'R²':    [r2_lr,   r2_rf,   r2_xgb,   r2_lgb]
}).sort_values('R²', ascending=False).reset_index(drop=True)

display(reg_results.round(4))

# Bar comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0']

for ax, metric in zip(axes, ['RMSE', 'MAE', 'R²']):
    ax.bar(reg_results['Model'], reg_results[metric], color=colors)
    ax.set_title(f'Regression — {metric}')
    ax.set_ylabel(metric)
    ax.tick_params(axis='x', rotation=20)
    if metric == 'R²':
        ax.set_ylim(0, 1)

plt.suptitle('Regression Model Comparison', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

best_reg = reg_results.iloc[0]['Model']
print(f"\n🏆 Best Regression Model: {best_reg}  (R² = {reg_results.iloc[0]['R²']:.4f})")


,Model,RMSE,MAE,R²
0,LightGBM,0.4974,0.2647,0.7405
1,XGBoost,0.4977,0.2674,0.7401
2,Linear Regression,0.5034,0.2950,0.7341
3,Random Forest,0.5449,0.2788,0.6885



🏆 Best Regression Model: LightGBM  (R² = 0.7405)


### 8.6 Residual Analysis (Best Regressor)

In [30]:
# Use XGBoost predictions for residual plot
residuals = yr_test - yr_pred_xgb

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(yr_pred_xgb, residuals, alpha=0.3, color='steelblue', s=10)
axes[0].axhline(0, color='red', linewidth=1.5)
axes[0].set_title('Residuals vs Predicted (XGBoost)')
axes[0].set_xlabel('Predicted Rating')
axes[0].set_ylabel('Residual')

axes[1].hist(residuals, bins=40, color='coral', edgecolor='white')
axes[1].axvline(0, color='navy', linestyle='--')
axes[1].set_title('Residual Distribution')
axes[1].set_xlabel('Residual')

plt.tight_layout()
plt.show()


### 8.7 Feature Importance — XGBoost Regressor

In [31]:
feat_imp = pd.Series(xgbr.feature_importances_, index=feature_cols).sort_values(ascending=False)

plt.figure(figsize=(10, 5))
feat_imp.plot(kind='bar', color='steelblue', edgecolor='white')
plt.title('Feature Importance — XGBoost Regressor')
plt.ylabel('Importance Score')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.show()

print("\nTop 5 most important features:")
print(feat_imp.head(5).to_string())



Top 5 most important features:
UserAvgRating          0.899070
AttractionAvgRating    0.032410
Season_enc             0.015266
UserVisitCount         0.012949
AttractionType_enc     0.007114


## Step 9 — Classification: Visit Mode Prediction

### 9.1 Random Forest Classifier

In [32]:
rfc = RandomForestClassifier(n_estimators=150, random_state=42, n_jobs=-1)
rfc.fit(X_train, yc_train)
yc_pred_rfc = rfc.predict(X_test)

acc_rfc = accuracy_score(yc_test, yc_pred_rfc)
f1_rfc  = f1_score(yc_test, yc_pred_rfc, average='weighted')
print("── Random Forest Classifier ────────────────")
print(f"  Accuracy : {acc_rfc:.4f}")
print(f"  F1-Score : {f1_rfc:.4f}")


── Random Forest Classifier ────────────────
  Accuracy : 0.4859
  F1-Score : 0.4754


### 9.2 XGBoost Classifier

In [33]:
xgbc = xgb.XGBClassifier(n_estimators=200, learning_rate=0.05,
                         max_depth=6, random_state=42, verbosity=0,
                         use_label_encoder=False, eval_metric='mlogloss')
xgbc.fit(X_train, yc_train)
yc_pred_xgbc = xgbc.predict(X_test)

acc_xgbc = accuracy_score(yc_test, yc_pred_xgbc)
f1_xgbc  = f1_score(yc_test, yc_pred_xgbc, average='weighted')
print("── XGBoost Classifier ──────────────────────")
print(f"  Accuracy : {acc_xgbc:.4f}")
print(f"  F1-Score : {f1_xgbc:.4f}")


── XGBoost Classifier ──────────────────────
  Accuracy : 0.4928
  F1-Score : 0.4401


### 9.3 LightGBM Classifier

In [34]:
lgbc = lgb.LGBMClassifier(n_estimators=200, learning_rate=0.05,
                          num_leaves=63, random_state=42, verbose=-1)
lgbc.fit(X_train, yc_train)
yc_pred_lgbc = lgbc.predict(X_test)

acc_lgbc = accuracy_score(yc_test, yc_pred_lgbc)
f1_lgbc  = f1_score(yc_test, yc_pred_lgbc, average='weighted')
print("── LightGBM Classifier ─────────────────────")
print(f"  Accuracy : {acc_lgbc:.4f}")
print(f"  F1-Score : {f1_lgbc:.4f}")


── LightGBM Classifier ─────────────────────
  Accuracy : 0.5126
  F1-Score : 0.4695


### 9.4 Gradient Boosting Classifier

In [35]:
gbc = GradientBoostingClassifier(n_estimators=150, learning_rate=0.1,
                                  max_depth=5, random_state=42)
gbc.fit(X_train, yc_train)
yc_pred_gbc = gbc.predict(X_test)

acc_gbc = accuracy_score(yc_test, yc_pred_gbc)
f1_gbc  = f1_score(yc_test, yc_pred_gbc, average='weighted')
print("── Gradient Boosting Classifier ────────────")
print(f"  Accuracy : {acc_gbc:.4f}")
print(f"  F1-Score : {f1_gbc:.4f}")


── Gradient Boosting Classifier ────────────
  Accuracy : 0.5010
  F1-Score : 0.4533


### 9.5 Classification Comparison

In [36]:
clf_results = pd.DataFrame({
    'Model': ['Random Forest', 'XGBoost', 'LightGBM', 'Gradient Boosting'],
    'Accuracy': [acc_rfc, acc_xgbc, acc_lgbc, acc_gbc],
    'F1-Score': [f1_rfc,  f1_xgbc,  f1_lgbc,  f1_gbc]
}).sort_values('F1-Score', ascending=False).reset_index(drop=True)

display(clf_results.round(4))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0']

for ax, metric in zip(axes, ['Accuracy', 'F1-Score']):
    ax.bar(clf_results['Model'], clf_results[metric], color=colors)
    ax.set_title(f'Classification — {metric}')
    ax.set_ylabel(metric)
    ax.set_ylim(0, 1)
    ax.tick_params(axis='x', rotation=20)

plt.suptitle('Classification Model Comparison', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

best_clf = clf_results.iloc[0]['Model']
print(f"\n🏆 Best Classification Model: {best_clf}  (F1 = {clf_results.iloc[0]['F1-Score']:.4f})")


,Model,Accuracy,F1-Score
0,Random Forest,0.4859,0.4754
1,LightGBM,0.5126,0.4695
2,Gradient Boosting,0.5010,0.4533
3,XGBoost,0.4928,0.4401



🏆 Best Classification Model: Random Forest  (F1 = 0.4754)


### 9.6 Detailed Classification Report & Confusion Matrix

In [37]:
# Use best model predictions
# Determine best based on f1
best_preds_map = {
    'Random Forest': yc_pred_rfc, 'XGBoost': yc_pred_xgbc,
    'LightGBM': yc_pred_lgbc, 'Gradient Boosting': yc_pred_gbc
}
best_preds = best_preds_map[clf_results.iloc[0]['Model']]

# Class labels
mode_le = le_dict['VisitModeName']
class_labels = mode_le.classes_

print("Detailed Classification Report:")
print("=" * 60)
print(classification_report(yc_test, best_preds,
                             target_names=[str(c) for c in class_labels]))


Detailed Classification Report:
              precision    recall  f1-score   support

    Business       0.44      0.22      0.30       125
     Couples       0.53      0.63      0.57      4324
      Family       0.50      0.50      0.50      3043
     Friends       0.38      0.30      0.33      2189
        Solo       0.38      0.23      0.28       905

    accuracy                           0.49     10586
   macro avg       0.44      0.38      0.40     10586
weighted avg       0.47      0.49      0.48     10586



In [38]:
# Confusion matrix
cm = confusion_matrix(yc_test, best_preds)
plt.figure(figsize=(10, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_labels, yticklabels=class_labels)
plt.title(f'Confusion Matrix — {clf_results.iloc[0]["Model"]}')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()


### 9.7 Feature Importance — LightGBM Classifier

In [39]:
fi_clf = pd.Series(lgbc.feature_importances_, index=feature_cols).sort_values(ascending=False)

plt.figure(figsize=(10, 5))
fi_clf.plot(kind='bar', color='mediumseagreen', edgecolor='white')
plt.title('Feature Importance — LightGBM Classifier')
plt.ylabel('Importance Score')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.show()


## Step 10 — Recommendation System

### 10.1 Collaborative Filtering (User-Item Matrix)

In [40]:
# ── Build user-item rating matrix ────────────────────────────────────
pivot = df.pivot_table(index='UserId', columns='AttractionId',
                       values='Rating', aggfunc='mean')
print(f"User-Item matrix shape: {pivot.shape}")
print(f"Sparsity: {(pivot.isnull().sum().sum() / pivot.size * 100):.1f}%")


User-Item matrix shape: (33530, 30)
Sparsity: 95.5%


In [41]:
# ── Matrix Factorization via Truncated SVD ───────────────────────────
# Fill NaN with user mean for SVD
pivot_filled = pivot.apply(lambda row: row.fillna(row.mean()), axis=1)

svd = TruncatedSVD(n_components=20, random_state=42)
U = svd.fit_transform(pivot_filled)   # User latent factors
S = np.diag(svd.singular_values_)
Vt = svd.components_                  # Item latent factors

# Reconstruct rating matrix
predicted_ratings = np.dot(U, Vt)

pred_df = pd.DataFrame(predicted_ratings,
                        index=pivot.index,
                        columns=pivot.columns)

print(f"Explained variance ratio (20 components): {svd.explained_variance_ratio_.sum():.3f}")
print("Predicted rating matrix (sample):")
display(pred_df.iloc[:3, :5].round(2))


Explained variance ratio (20 components): 1.000
Predicted rating matrix (sample):


AttractionId,369,481,640,650,673
UserId,,,,,
14,4.50,4.5,4.00,4.50,4.50
16,4.85,5.0,4.25,4.85,4.85
20,4.00,4.0,4.00,4.00,4.00


In [42]:
def recommend_collaborative(user_id, top_n=10):
    """Recommend top_n attractions for a given user using SVD collaborative filtering."""
    if user_id not in pred_df.index:
        return pd.DataFrame({'Message': [f'User {user_id} not found in training data']})

    user_row = pred_df.loc[user_id]

    # Attractions already visited
    visited = pivot.loc[user_id].dropna().index.tolist()

    # Sort unvisited by predicted score
    recommendations = (user_row.drop(index=visited, errors='ignore')
                                .sort_values(ascending=False)
                                .head(top_n))

    result = pd.DataFrame({
        'AttractionId': recommendations.index,
        'PredictedRating': recommendations.values
    })

    # Merge attraction details
    result = result.merge(item[['AttractionId', 'Attraction', 'AttractionTypeId']],
                          on='AttractionId', how='left')
    result = result.merge(atype, on='AttractionTypeId', how='left')
    return result[['AttractionId', 'Attraction', 'AttractionType', 'PredictedRating']]

# Demo
sample_user = df['UserId'].value_counts().index[0]
print(f"Collaborative Filtering Recommendations for User {sample_user}:")
display(recommend_collaborative(sample_user, top_n=10))


Collaborative Filtering Recommendations for User 60799:


,AttractionId,Attraction,AttractionType,PredictedRating
0,975,Sempu Island,Nature & Wildlife Areas,3.710909
1,947,Mount Semeru Volcano,Volcanos,3.704038
2,920,Jodipan Colorful Village,Neighborhoods,3.701827
3,877,Balekambang Beach,Beaches,3.701621
4,937,Malang City Square,Points of Interest & Landmarks,3.701536
5,928,Khayangan Reflexology & Massage,Spas,3.701521
6,897,Coban Rondo Waterfall,Waterfalls,3.701437
7,949,Museum Malang Tempo Doeloe,Speciality Museums,3.701228
8,1278,Ullen Sentalu Museum,History Museums,3.701130
9,1280,Water Castle (Tamansari),Points of Interest & Landmarks,3.700923


### 10.2 Content-Based Filtering

In [43]:
# ── Build attraction feature matrix ──────────────────────────────────
att_features = item[['AttractionId', 'AttractionTypeId', 'AttractionCityId']].copy()
att_features = att_features.merge(
    df.groupby('AttractionId')['Rating'].mean().rename('AvgRating'),
    on='AttractionId', how='left').fillna(0)
att_features = att_features.merge(
    df.groupby('AttractionId')['TransactionId'].count().rename('Popularity'),
    on='AttractionId', how='left').fillna(0)

att_feature_matrix = att_features[['AttractionTypeId', 'AttractionCityId',
                                    'AvgRating', 'Popularity']].values
scaler_att = StandardScaler()
att_feature_matrix_s = scaler_att.fit_transform(att_feature_matrix)

# Fit KNN for similarity
knn_att = NearestNeighbors(n_neighbors=11, metric='cosine', algorithm='brute')
knn_att.fit(att_feature_matrix_s)

att_id_to_idx = {aid: idx for idx, aid in enumerate(att_features['AttractionId'])}

def recommend_content_based(attraction_id, top_n=10):
    """Find top_n attractions similar to the given attraction."""
    if attraction_id not in att_id_to_idx:
        return pd.DataFrame({'Message': [f'Attraction {attraction_id} not found']})
    idx = att_id_to_idx[attraction_id]
    dists, idxs = knn_att.kneighbors([att_feature_matrix_s[idx]])
    similar_ids = att_features.iloc[idxs[0][1:]]['AttractionId'].values

    result = item[item['AttractionId'].isin(similar_ids)][['AttractionId', 'Attraction', 'AttractionTypeId']].copy()
    result = result.merge(atype, on='AttractionTypeId', how='left')
    result['SimilarityDistance'] = dists[0][1:len(result)+1].round(4)
    return result[['AttractionId', 'Attraction', 'AttractionType', 'SimilarityDistance']]

# Demo
sample_att = df['AttractionId'].value_counts().index[0]
att_name = item[item['AttractionId']==sample_att]['Attraction'].values[0]
print(f"Content-Based Recommendations similar to: '{att_name}' (ID={sample_att})")
display(recommend_content_based(sample_att, top_n=10))


Content-Based Recommendations similar to: 'Sacred Monkey Forest Sanctuary' (ID=640)


,AttractionId,Attraction,AttractionType,SimilarityDistance
0,369,Kuta Beach - Bali,Beaches,0.1049
1,481,Nusa Dua Beach,Beaches,0.2441
2,650,Sanur Beach,Beaches,0.3753
3,673,Seminyak Beach,Beaches,0.3760
4,737,Tanah Lot Temple,Religious Sites,0.6514
5,748,Tegalalang Rice Terrace,Points of Interest & Landmarks,0.7302
6,749,Tegenungan Waterfall,Waterfalls,0.7395
7,824,Uluwatu Temple,Religious Sites,0.7666
8,841,Waterbom Bali,Water Parks,0.8835
9,1171,Merapi Volcano,Volcanos,1.0455


### 10.3 Hybrid Recommendation System

In [44]:
def hybrid_recommend(user_id, top_n=10, alpha=0.6):
    """
    Hybrid = alpha * Collaborative Score  +  (1-alpha) * Content-Based Score
    alpha = 0.6 gives more weight to collaborative filtering.
    """
    if user_id not in pred_df.index:
        return recommend_content_based(df['AttractionId'].mode()[0], top_n)

    # Collaborative scores
    visited = pivot.loc[user_id].dropna().index.tolist()
    collab_scores = pred_df.loc[user_id].drop(index=visited, errors='ignore')

    # Content-based scores: average KNN distance over top-3 visited attractions
    user_visited = df[df['UserId'] == user_id].nlargest(3, 'Rating')['AttractionId'].tolist()
    content_scores = {}
    for av in user_visited:
        if av in att_id_to_idx:
            idx = att_id_to_idx[av]
            dists, idxs = knn_att.kneighbors([att_feature_matrix_s[idx]])
            for d, i in zip(dists[0][1:], idxs[0][1:]):
                aid = att_features.iloc[i]['AttractionId']
                # Convert distance to similarity
                content_scores[aid] = content_scores.get(aid, 0) + (1 - d)

    # Normalize and combine
    cs = pd.Series(content_scores)
    cs = cs / cs.max() if cs.max() > 0 else cs
    collab_norm = (collab_scores - collab_scores.min()) / (collab_scores.max() - collab_scores.min() + 1e-6)

    combined = alpha * collab_norm + (1 - alpha) * cs
    combined = combined.sort_values(ascending=False).head(top_n)

    result = pd.DataFrame({'AttractionId': combined.index, 'HybridScore': combined.values})
    result = result.merge(item[['AttractionId', 'Attraction', 'AttractionTypeId']], on='AttractionId', how='left')
    result = result.merge(atype, on='AttractionTypeId', how='left')
    return result[['AttractionId', 'Attraction', 'AttractionType', 'HybridScore']].round({'HybridScore': 4})

print(f"Hybrid Recommendations for User {sample_user}:")
display(hybrid_recommend(sample_user, top_n=10))


Hybrid Recommendations for User 60799:


,AttractionId,Attraction,AttractionType,HybridScore
0,841.0,Waterbom Bali,Water Parks,0.4821
1,824.0,Uluwatu Temple,Religious Sites,0.4234
2,737.0,Tanah Lot Temple,Religious Sites,0.4232
3,947.0,Mount Semeru Volcano,Volcanos,0.3603
4,1280.0,Water Castle (Tamansari),Points of Interest & Landmarks,0.3167
5,897.0,Coban Rondo Waterfall,Waterfalls,0.3032
6,949.0,Museum Malang Tempo Doeloe,Speciality Museums,0.2861
7,1278.0,Ullen Sentalu Museum,History Museums,0.2669
8,749.0,Tegenungan Waterfall,Waterfalls,0.2646
9,928.0,Khayangan Reflexology & Massage,Spas,0.2422


### 10.4 Recommendation System Evaluation (RMSE on held-out ratings)

In [45]:
# Evaluate collaborative filtering RMSE on test set
# Sample 2000 known ratings not used in SVD training
eval_sample = df[['UserId', 'AttractionId', 'Rating']].sample(2000, random_state=42)

actual, predicted = [], []
for _, row in eval_sample.iterrows():
    uid = row['UserId']
    aid = row['AttractionId']
    if uid in pred_df.index and aid in pred_df.columns:
        actual.append(row['Rating'])
        predicted.append(pred_df.loc[uid, aid])

actual    = np.array(actual)
predicted = np.array(predicted)

rec_rmse = np.sqrt(mean_squared_error(actual, predicted))
rec_mae  = mean_absolute_error(actual, predicted)

print("── Recommendation System (Collaborative SVD) Evaluation ──")
print(f"  RMSE : {rec_rmse:.4f}")
print(f"  MAE  : {rec_mae:.4f}")
print(f"  Evaluated on {len(actual)} known (user, attraction) pairs")


── Recommendation System (Collaborative SVD) Evaluation ──
  RMSE : 0.2290
  MAE  : 0.0646
  Evaluated on 2000 known (user, attraction) pairs


## Step 11 — Summary & Business Insights

In [46]:
print("=" * 65)
print("  PROJECT SUMMARY — Tourism Experience Analytics")
print("=" * 65)

print("\n📊 DATASET STATISTICS")
print(f"   Total Transactions  : {len(transaction):,}")
print(f"   Unique Users        : {df['UserId'].nunique():,}")
print(f"   Unique Attractions  : {df['AttractionId'].nunique()}")
print(f"   Attraction Types    : {df['AttractionType'].nunique()}")
print(f"   Overall Avg Rating  : {df['Rating'].mean():.3f}")

print("\n🔵 REGRESSION — Predicting Attraction Ratings")
print(f"   Best Model  : {reg_results.iloc[0]['Model']}")
print(f"   R²          : {reg_results.iloc[0]['R²']:.4f}")
print(f"   RMSE        : {reg_results.iloc[0]['RMSE']:.4f}")
print(f"   MAE         : {reg_results.iloc[0]['MAE']:.4f}")

print("\n🟢 CLASSIFICATION — Predicting Visit Mode")
print(f"   Best Model  : {clf_results.iloc[0]['Model']}")
print(f"   Accuracy    : {clf_results.iloc[0]['Accuracy']:.4f}")
print(f"   F1-Score    : {clf_results.iloc[0]['F1-Score']:.4f}")

print("\n🟡 RECOMMENDATION — SVD Collaborative Filtering")
print(f"   RMSE on held-out : {rec_rmse:.4f}")
print(f"   MAE  on held-out : {rec_mae:.4f}")


  PROJECT SUMMARY — Tourism Experience Analytics

📊 DATASET STATISTICS
   Total Transactions  : 52,930
   Unique Users        : 33,530
   Unique Attractions  : 30
   Attraction Types    : 17
   Overall Avg Rating  : 4.158

🔵 REGRESSION — Predicting Attraction Ratings
   Best Model  : LightGBM
   R²          : 0.7405
   RMSE        : 0.4974
   MAE         : 0.2647

🟢 CLASSIFICATION — Predicting Visit Mode
   Best Model  : Random Forest
   Accuracy    : 0.4859
   F1-Score    : 0.4754

🟡 RECOMMENDATION — SVD Collaborative Filtering
   RMSE on held-out : 0.2290
   MAE  on held-out : 0.0646


In [47]:
# ── Business Insight Visuals ─────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# 1. Top continents by avg rating
cont_rating = df.groupby('ContinentName')['Rating'].mean().sort_values(ascending=False)
axes[0].bar(cont_rating.index, cont_rating.values, color=sns.color_palette('Set2', len(cont_rating)))
axes[0].set_title('Avg Rating by Continent')
axes[0].set_ylabel('Avg Rating')
axes[0].set_ylim(0, 5.5)
axes[0].tick_params(axis='x', rotation=30)

# 2. Visit mode distribution
mode_counts = df['VisitModeName'].value_counts()
axes[1].pie(mode_counts, labels=mode_counts.index, autopct='%1.1f%%',
            colors=sns.color_palette('pastel', len(mode_counts)))
axes[1].set_title('Visit Mode Distribution')

# 3. Year-over-year transaction growth
yr_trend = df.groupby('VisitYear').size()
axes[2].plot(yr_trend.index, yr_trend.values, marker='o', color='steelblue', linewidth=2)
axes[2].fill_between(yr_trend.index, yr_trend.values, alpha=0.2, color='steelblue')
axes[2].set_title('Year-over-Year Transactions')
axes[2].set_xlabel('Year')
axes[2].set_ylabel('Transactions')

plt.suptitle('Key Business Insights', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


## Step 12 — Key Business Insights & Actionable Recommendations

| # | Insight | Recommendation |
|---|---------|----------------|
| 1 | **Attraction popularity is highly skewed** — a few attractions attract most visits | Invest in improving mid-tier attractions to distribute traffic |
| 2 | **Couples and Family modes** dominate transactions | Launch targeted family & couples packages to maximise bookings |
| 3 | **UserAvgRating & AttractionAvgRating** are the strongest predictors | Personalize suggestions based on user rating history |
| 4 | **Summer & Winter** see peak visits | Offer seasonal discounts in Spring & Autumn to balance load |
| 5 | **SVD collaborative filtering** achieves good recommendation RMSE | Deploy the hybrid recommender for higher precision |
| 6 | **Region and Attraction Type** are powerful classification features | Use region-specific marketing to boost user retention |

---
*This notebook covers: Data Cleaning → Preprocessing → Feature Engineering → EDA → Regression → Classification → Recommendation → Evaluation → Insights*
